# 11 Lookup Memory: N-Gram Priors and Engram-Style Residuals

After reading Tutorials 00, 07, and 09, let's examine inference-time algorithms that look up tables keyed by context tokens. Two representative examples are modifying logits directly with an n-gram language model prior (or kNN-LM-style adjustments), and DeepSeek's Engram—hashing n-grams into a large embedding table, applying a gate, and adding the result into the residual stream.

Bringing such algorithms into an inference engine raises three practical questions:
1. **How to obtain context tokens**: lookup keys depend solely on token IDs, not hidden states. Inside an inference engine, you need each row's current token ID and its preceding tokens—and they must remain accurate under continuous batching, chunked prefill, and preemption recomputation.
2. **Where to place large tables**: these tables are often huge (an Engram table can exceed the size of the model itself), so you must decide whether they live in GPU VRAM or host memory.
3. **How expensive PCIe transfers are**: when a large table sits in host memory, every step pulls rows over PCIe. You want to see how many bytes move per step and how much throughput drops before running anything.

In pLLM, these three pieces are expressed as follows:
- `pllm.input_tokens(history=n)` gives each row `[current token, previous, ..., n back]` (`int64`, padded with `-1` before the start of a request). Data comes directly from vLLM's GPU-resident token history with zero GPU-to-CPU synchronization.
- Tables are loaded from safetensors via `InstanceState(init=pllm.Pretrained(file))`. Placement is explicit: `pllm.Pretrained(..., placement="device")` (default; stays in VRAM and never silently spills to host) or `placement="host"` (pinned host memory read directly per row by the GPU). pLLM never silently moves a table—that is an expensive choice made explicitly by the author, with costs itemized by `pllm.explain`.
- The same lookup result can feed either `@pllm.add_logits` (n-gram prior) or `@pllm.custom_residual` (Engram-style residual).

Before starting, note these limitations and costs in this example:
- The table here is counted from a tiny synthetic corpus rather than trained; `scale=10` causes the model to recite the corpus, which only shows that table lookup is wired up, not a quality improvement.
- In the Engram-style residual step, the table stores next-token word embeddings from the corpus; adding them into a model never trained for that architecture has only a weak effect. Real Engram trains the table and gate jointly with the model.
- Throughput has a cost: the logits prior adds a vocab-wide vector per row; on 7B in this run, `device` placement reaches `98%` of vanilla vLLM and `host` placement reaches `96%`, a modest additional cost (measured in Step 5).
- Outputs and numbers below come from RTX 4090, vLLM 0.32.0; Steps 1–4 and 6–7 use `Qwen/Qwen2.5-0.5B-Instruct`, while Step 5 throughput uses `Qwen/Qwen2.5-7B-Instruct` (bf16).

We proceed in seven steps:
1. Inspect what `pllm.input_tokens` returns;
2. Count an n-gram table from a short corpus;
3. Write the n-gram prior workload and compare device vs. host placement with `explain`;
4. Run contract checks and text generation;
5. Benchmark throughput;
6. Feed the same lookup into the residual stream (Engram style);
7. Run GPU contract checks.

First, import dependencies and prepare the model configuration.

In [1]:
import os

os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"  # Show only vLLM errors to suppress startup logs

import tempfile
from collections import defaultdict
from dataclasses import dataclass
from typing import Literal

import torch
from safetensors.torch import save_file
from transformers import AutoTokenizer
from vllm import SamplingParams

import pllm
from pllm import testing
from pllm.benchmark import throughput

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
CONFIG = pllm.ModelConfig(MODEL, max_model_len=512, max_num_seqs=8, max_num_batched_tokens=512,
                          gpu_memory_utilization=0.4, seed=0)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

## Step 1: What `pllm.input_tokens` Returns

Before writing the lookup logic, let's verify what tokens each row actually receives via `pllm.input_tokens` during prefill and decode.

Below we write a small export-only workload `ShowTokens`. It reads `[current token, previous, two back]` per row via `pllm.input_tokens(history=2)` and casts it to `.float()` for host export.

Keep this rule in mind for what each row represents:
- If a computation feeds into `custom_residual`, it runs for **every scheduled token** (including every prompt token).
- Other methods (such as `@pllm.export` here and `@pllm.add_logits` later) run for **sampled rows**, i.e., the last token of the prompt and each decode token generated thereafter.

Let's run it to inspect the token IDs.

In [2]:
seen: list[list[int]] = []


@dataclass(frozen=True)
class ShowTokens(pllm.Workload):
    @pllm.export(name="ids", max_rows=4, max_cols=3)
    def ids(self, ids: torch.Tensor = pllm.input_tokens(history=2)) -> torch.Tensor:
        return ids.float()  # [rows, 3]: current token, previous, two tokens back

    def on_export(self, event: pllm.io.TraceEvent) -> None:
        seen.extend(event.tensor().long().tolist())


with pllm.make_llm(CONFIG, workload=ShowTokens()) as llm:
    output = llm.generate(["The capital of France is"], SamplingParams(max_tokens=4, temperature=0.0),
                          use_tqdm=False)[0]
print("prompt 的 token:", [tokenizer.decode([t]) for t in output.prompt_token_ids])
print("生成的 token:  ", [tokenizer.decode([t]) for t in output.outputs[0].token_ids])
for row in seen:
    print(row, "->", [tokenizer.decode([t]) for t in row])

prompt 的 token: ['The', ' capital', ' of', ' France', ' is']
生成的 token:   [' Paris', '.', ' It', ' is']
[374, 9625, 315] -> [' is', ' France', ' of']
[12095, 374, 9625] -> [' Paris', ' is', ' France']
[13, 12095, 374] -> ['.', ' Paris', ' is']
[1084, 13, 12095] -> [' It', '.', ' Paris']


Four rows were exported, corresponding to four sampling steps:
- Row 1 `[374, 9625, 315]` -> `[' is', ' France', ' of']`: the current token is the prompt's final token `' is'` (whose output samples the first generated token `' Paris'`), preceded by `' France'` and `' of'`.
- Row 2 `[12095, 374, 9625]` -> `[' Paris', ' is', ' France']`: the current token is now `' Paris'` generated in the previous step, preceded by `' is'` and `' France'`.
- On each subsequent step, the current token is the token generated in the previous step. Four tokens were generated in total; the 4th token `' is'` was sampled by row 4, after which the request finished without being fed back as an input, so it does not appear in the exported table.

## Step 2: Counting an N-Gram Table from a Corpus

Having verified the ordering of `pllm.input_tokens`, let's build a table to query. We want to see clearly whether the model retrieves next-token bonuses when encountering familiar bigram contexts, while also making the table large enough to show the difference between VRAM and host-memory placement.

We prepare a 4-sentence synthetic corpus describing a fictional festival, the "Lantern Fair of Quillmere." At every position in the corpus, we use `(current token, previous token)` as the context and count which tokens follow and how often.

Table layout:
- `2048` buckets total, with contexts mapped to bucket indices by a simple integer hash `ngram_keys`.
- Each bucket stores two fields:
  1. `context`: the 2 `int64` token IDs corresponding to that bucket, checked on lookup to prevent false hits from hash collisions.
  2. `bonus`: a `bfloat16` logits bonus vector of width `151936` (Qwen2.5's LM head width, including padding). Next tokens seen in the corpus receive `1 + log(count)`, and all other entries are `0`.

We save both tensors into a single safetensors file.

In [3]:
corpus = (
    "The Lantern Fair of Quillmere is held every year on the first full moon of autumn. "
    "Families carry paper lanterns shaped like herons down to the Silverbrook river, and the oldest baker in town "
    "hands out honey cakes stamped with a crescent moon. The fair ends at midnight, when the lanterns are set afloat "
    "and the bell of the old granary rings thirteen times."
)
BUCKETS = 2048
VOCAB = 151936  # Qwen2.5 LM head width (including padding)


def ngram_keys(current: torch.Tensor, previous: torch.Tensor, buckets: int) -> torch.Tensor:
    return (current * 1000003 + previous + 1).remainder(buckets)


def fit_table(text: str, buckets: int, path: str, vocab: int = VOCAB) -> int:
    """Count what follows each bigram in the corpus and write a hash table."""
    ids = tokenizer(text).input_ids
    follows: dict[tuple[int, int], dict[int, int]] = defaultdict(lambda: defaultdict(int))
    for t in range(1, len(ids) - 1):
        follows[(ids[t], ids[t - 1])][ids[t + 1]] += 1
    context = torch.full((buckets, 2), -2, dtype=torch.int64)  # Context stored per bucket for collision checks
    bonus = torch.zeros(buckets, vocab, dtype=torch.bfloat16)  # One logits bonus row per bucket
    for (current, previous), nexts in follows.items():
        key = int(ngram_keys(torch.tensor(current), torch.tensor(previous), buckets))
        context[key] = torch.tensor([current, previous])
        bonus[key] = 0
        for token, count in nexts.items():
            bonus[key, token] = 1.0 + torch.log(torch.tensor(float(count)))
    save_file({"context": context, "bonus": bonus}, path)
    return len(follows)


TABLE = os.path.join(tempfile.gettempdir(), "lantern_ngram.safetensors")
print("语料", len(tokenizer(corpus).input_ids), "个 token，", fit_table(corpus, BUCKETS, TABLE), "个不同的上下文；",
      f"表文件 {os.path.getsize(TABLE) / 2**20:.0f} MiB")

语料 78 个 token， 74 个不同的上下文； 表文件 594 MiB


The corpus has `78` tokens and `74` distinct contexts, producing a `594 MiB` table file. Since only `74` buckets are populated, most rows in this table are zeros—we intentionally sized it to `2048 × 151936` so the next step clearly contrasts device vs. host placement.

## Step 3: Authoring the N-Gram Prior and Comparing Placement

Now we write the `NGramPrior` workload: at each step, look up the table using the current and previous token; if the bucket's stored context matches the row, extract that `151936`-wide bonus row, multiply by `scale`, and add it to the logits.

Two patterns in the code are worth noting:
1. **One `InstanceState` holds the whole file**: `table` declares two fields loaded by name from the safetensors file via `init=pllm.Pretrained(self.path)`. `context` needs `int64`, so it uses `pllm.Field((self.buckets, 2), torch.int64)`; `bonus` matches the model's vocabulary width, written as a shape function `lambda m: (self.buckets, m.vocab_size)` where `m` holds model dimensions. `placement` applies to the entire state: with `"host"`, both `context` and `bonus` live in host memory.
2. **Intermediate result `lookup`**: `lookup` itself has no decorator. It reads `pllm.input_tokens(history=1)`, hashes to bucket keys, checks `context == ids`, and returns `bonus[keys].float() * self.scale` on hits. `prior` (`@pllm.add_logits()`) then consumes this intermediate result via `pllm.result("lookup")` and adds it to logits.

Before launching on GPU, let's compare the resource accounting of `placement="device"` and `placement="host"` on CPU using `pllm.explain`.

In [4]:
@dataclass(frozen=True)
class NGramPrior(pllm.Workload):
    path: str = TABLE
    scale: float = 0.0
    buckets: int = BUCKETS
    placement: Literal["device", "host"] = "device"

    @property
    def table(self) -> pllm.InstanceState:
        return pllm.InstanceState({"context": pllm.Field((self.buckets, 2), torch.int64),
                                   "bonus": lambda m: (self.buckets, m.vocab_size)},
                                  init=pllm.Pretrained(self.path, placement=self.placement))

    def lookup(self, ids: torch.Tensor = pllm.input_tokens(history=1),
               context: torch.Tensor = pllm.parameter("table", "context"),
               bonus: torch.Tensor = pllm.parameter("table", "bonus")) -> torch.Tensor:
        keys = ngram_keys(ids[:, 0], ids[:, 1], self.buckets)
        hit = (context[keys] == ids).all(-1, keepdim=True)  # Hit only if the stored bucket context matches this row
        return torch.where(hit, bonus[keys].float(), 0.0) * self.scale

    @pllm.add_logits()
    def prior(self, extra: torch.Tensor = pllm.result("lookup")) -> torch.Tensor:
        return extra

In [5]:
for placement in ("device", "host"):
    report = pllm.explain(NGramPrior(scale=10.0, placement=placement), CONFIG)
    for line in str(report).splitlines():
        if line.startswith(("table ", "memory ")):
            print(line.replace(f"pretrained {TABLE}", "pretrained <TABLE>"))
    print()

table  instance  622362624  622362638  field 0: int64 x 1 pages, field 1: bfloat16 x 1 pages; source: pretrained <TABLE> sha256 45f6aaccc61895503b2f82d8ac0b3be7e29afeacec37e3387b0c8bb0d4a8c492 (622362784 bytes); placement: device; host bytes: 0; logical transfer bytes/row: 0; decode bytes/step: 0
memory   peak 736516646 bytes funded by pLLM [per-rank tensor reservations; excludes vLLM model-graph/activation pools and GPU-dependent BLAS workspace; execution_peak_budget_bytes]

table  instance  622362624  0  field 0: int64 x 1 pages, field 1: bfloat16 x 1 pages; source: pretrained <TABLE> sha256 45f6aaccc61895503b2f82d8ac0b3be7e29afeacec37e3387b0c8bb0d4a8c492 (622362784 bytes); placement: host; host bytes: 622362638; logical transfer bytes/row: 303888; decode bytes/step: 2431104
memory   peak 114154008 bytes funded by pLLM [per-rank tensor reservations; excludes vLLM model-graph/activation pools and GPU-dependent BLAS workspace; execution_peak_budget_bytes]



Let's check both `explain` reports line by line:
- **With `placement="device"` (table in VRAM)**: `table` reserves `622362638` bytes in VRAM, with `host bytes: 0`, `logical transfer bytes/row: 0`, and `decode bytes/step: 0`. Total workload peak VRAM reservation (`memory peak`) is `736516646` bytes.
- **With `placement="host"` (table in host memory)**: `table` VRAM reservation drops to `0`, while host memory usage is `622362638` bytes. `explain` also calculates PCIe transfer volume: `logical transfer bytes/row: 303888` (`303888 = 151936 × 2 + 2 × 8` bytes, matching one `151936`-wide BF16 bonus row plus two `int64` context IDs), giving `decode bytes/step: 2431104` across 8 decode rows at `max_num_seqs=8`. Correspondingly, pLLM's peak VRAM reservation drops to `114154008` bytes.
- **File identity tracking**: `source` records the file path and its `sha256` (`45f6aaccc61895503b2f82d8ac0b3be7e29afeacec37e3387b0c8bb0d4a8c492`). If the underlying table file changes, the workload's identity changes with it.

Note that `explain` reports byte counts and does not convert them to latency; Step 5 measures how much those transfers slow down actual generation.

## Step 4: Contract Checks and Text Generation

Before running in vLLM, we verify on CPU with `testing.check` that the workload satisfies all batching contracts (row permutation invariance, request isolation, mixed prefill/decode, chunked prefill, preemption recomputation, and equivalence to the uninstrumented model when `scale=0`).

Below we run `testing.check(NGramPrior(scale=10.0), CONFIG, neutral=NGramPrior(scale=0.0))` directly against the `594 MiB` (`2048`-bucket) table with `scale=0.0` as `neutral`; on CPU this takes ~`28` seconds with ~`6.6 GiB` peak process memory.

In [6]:
print(testing.check(NGramPrior(scale=10.0), CONFIG, neutral=NGramPrior(scale=0.0)))

status  contract / reason
PASS    tensor_parallel_size=1: admission
PASS    compile
PASS    row permutation
PASS    request isolation
PASS    mixed prefill/decode
PASS    chunking / tokens=
PASS    publish_lag
PASS    preempted
PASS    neutral


All 9 contracts `PASS`. In particular, `mixed prefill/decode`, `chunking / tokens=`, and `preempted` verify that when prompts are chunked, mixed with decode rows in a batch, or preempted and recomputed, every row's token history from `pllm.input_tokens(history=1)` stays identical.

With contracts passing, we generate text on GPU using the `2048`-bucket (`594 MiB`) table. We ask two questions about the Lantern Fair using greedy decoding (`temperature=0.0`) for `40` tokens across four configurations:
1. Vanilla vLLM (no workload);
2. `scale=0.0` (workload attached, bonus scaled by 0);
3. `scale=10.0` with table in VRAM (`placement="device"`);
4. `scale=10.0` with table in host memory (`placement="host"`).

In [7]:
questions = ["Tell me about the Lantern Fair of Quillmere.", "What happens at the end of the Lantern Fair?"]
prompts = [tokenizer.apply_chat_template([{"role": "user", "content": q}], tokenize=False, add_generation_prompt=True)
           for q in questions]
greedy = SamplingParams(max_tokens=40, temperature=0.0)


def answers(llm) -> list[str]:
    return [output.outputs[0].text for output in llm.generate(prompts, greedy, use_tqdm=False)]


with pllm.make_llm(CONFIG) as plain:
    baseline = answers(plain)
for text in baseline:
    print("原生 vLLM:", repr(text))
results = {}
for workload in (NGramPrior(scale=0.0), NGramPrior(scale=10.0), NGramPrior(scale=10.0, placement="host")):
    with pllm.make_llm(CONFIG, workload=workload) as llm:
        results[(workload.scale, workload.placement)] = answers(llm)
print("scale=0 与原生 vLLM 逐字相同:", results[(0.0, "device")] == baseline)
for text in results[(10.0, "device")]:
    print("scale=10:", repr(text))
print("表放在主机内存与放在显存，输出逐字相同:", results[(10.0, "host")] == results[(10.0, "device")])

原生 vLLM: 'The Lantern Fair of Quillmere is a traditional festival celebrated in the small town of Quillmere, located in the United States. This event is a significant cultural and economic event that has been held'
原生 vLLM: 'The Lantern Fair is a traditional Chinese festival that celebrates the end of the Lantern Festival. The end of the Lantern Fair typically marks the end of the festival and the beginning of the New Year. Here are'


scale=0 与原生 vLLM 逐字相同: True
scale=10: 'The Lantern Fair of Quillmere is held every year on the first full moon of autumn. Families carry paper lanterns shaped like herons down to the riverbank to light them. The fair ends'
scale=10: 'The Lantern Fair of Quillmere is held every year on the first full moon of autumn. Families carry paper lanterns shaped like herons down to the river to light them. The fair ends at'
表放在主机内存与放在显存，输出逐字相同: True


Inspect the generated responses:
- **Vanilla vLLM**: because the model has never seen our fictional town of Quillmere, it invents `"a small town, located in the United States"` for the first question and answers `"traditional Chinese festival"` for the second.
- **`scale=0`**: output matches vanilla vLLM verbatim (`True`).
- **`scale=10`**: both answers open by reciting our synthetic corpus: `"The Lantern Fair of Quillmere is held every year on the first full moon of autumn. Families carry paper lanterns shaped like herons down to the"`. They then diverge into `"riverbank to light them."` and `"river to light them."` (where the corpus had `"Silverbrook river"`), before returning to `"The fair ends"`.
- **Host vs. device placement**: both produce verbatim identical outputs (`True`): placement affects only speed and VRAM, never numerical results.

To reiterate: this table was counted from a tiny synthetic corpus rather than trained; `scale=10` reciting the corpus shows that token-history lookup and logit modification work end-to-end, not an improvement in generation quality.

## Step 5: Measuring Throughput Cost

Earlier, `explain` showed that device placement avoids PCIe transfers while still processing a `151936`-wide vector per row, whereas host placement also transfers `303888` bytes per row over PCIe. Let's measure how much each slows down execution and whether PCIe transfers can be hidden.

0.5B sufficed for earlier steps; for throughput we switch to `Qwen/Qwen2.5-7B-Instruct` because a 0.5B decode step takes only a few milliseconds (amplifying lookup/transfer overhead), whereas 7B reflects typical deployment scale (bf16 weights ~15 GiB, requiring a 24 GiB GPU). 7B's LM head has width `152064`, so we build a `2048 × 152064` table (also ~594 MiB) from the same corpus, moving `152064 × 2 + 2 × 8 = 304144` bytes per row over PCIe.

Benchmark setup: `128` concurrent requests (`8` prompts × `16` samples × `256` tokens, `ignore_eos=True`), measured via `pllm.benchmark.throughput`: 1 warmup round followed by the median of 3 timed rounds, timing the entire `generate` call (including prefill).

In [8]:
many = [f"Explain the mathematical idea numbered {i}:" for i in range(8)]
sampling = SamplingParams(n=16, max_tokens=256, temperature=0.8, top_p=0.95, ignore_eos=True, seed=7)
BIG = pllm.ModelConfig("Qwen/Qwen2.5-7B-Instruct", max_model_len=512, max_num_seqs=128, max_num_batched_tokens=512,
                       gpu_memory_utilization=0.8, seed=0)
BIG_TABLE = os.path.join(tempfile.gettempdir(), "lantern_ngram_7b.safetensors")
fit_table(corpus, BUCKETS, BIG_TABLE, vocab=152064)  # Qwen2.5-7B LM head width

rates = {}
with pllm.make_llm(BIG) as plain:
    rates["原生 vLLM"] = throughput(plain, many, sampling)
for placement in ("device", "host"):
    with pllm.make_llm(BIG, workload=NGramPrior(BIG_TABLE, scale=10.0, placement=placement)) as llm:
        rates[f"表在 {placement}"] = throughput(llm, many, sampling)
for name, rate in rates.items():
    print(f"{name:10s}: {rate:6.0f} token/s，为原生 vLLM 的 {rate / rates['原生 vLLM']:.0%}")

原生 vLLM   :   5877 token/s，为原生 vLLM 的 100%
表在 device :   5742 token/s，为原生 vLLM 的 98%
表在 host   :   5615 token/s，为原生 vLLM 的 96%


Measured results (7B, 128 concurrent requests; ~128 rows per decode step):
- **Vanilla vLLM**: `5877 token/s` (`100%`), ~`21.8 ms` per decode step.
- **Table on `device` (VRAM)**: `5742 token/s`, `98%` of vanilla vLLM (~`22.3 ms`/step). This cost comes from fetching a `152064`-wide BF16 bonus row per sampled row, scaling it, and adding it to logits.
- **Table on `host` (host memory)**: `5615 token/s`, `96%` of vanilla vLLM (~`22.8 ms`/step). On top of `device`, each row reads `304144` bytes over PCIe; `host` adds only ~`0.5 ms` per step over `device`, even though moving `128 × 304144 ≈ 38.9 MB` at the measured ~`23.7 GB/s` takes ~`1.6 ms`—meaning most of the transfer overlaps with the model's forward pass.

Compared to 0.5B, a 7B decode step is much longer (streaming 15 GB of weights), so the same 38.9 MB lookup and transfer is a smaller fraction of step time, shrinking the relative overhead of both placements.

Why does overlap work? When a lookup method depends only on `pllm.input_tokens` and read-only tables, pLLM schedules it at the start of the step (`request.prepare`) on a `side` CUDA stream concurrently with the model forward. When reading host tables, pLLM uses only a small fraction of SMs (~1/4), because host-memory reads spend most of their time waiting on PCIe and a few thread blocks saturate PCIe bandwidth, leaving the remaining SMs for the forward pass. In `pllm.explain`, such a method shows up as `lookup  compute  request.prepare / side / captured` (as seen on `NGramPrior` here), allowing authors to verify side-stream overlap directly. Row-wise reads on host tables—`table[idx]`, `torch.index_select`, `torch.nn.functional.embedding`, row-wise indexing inside `torch.vmap`, and slicing a view (such as `table[:, ::2]`) before row indexing—all take this path; other operations on host tables are rejected during planning.

This yields a simple rule of thumb: PCIe bytes per step = rows × `logical transfer bytes/row` from `explain`, divided by PCIe bandwidth. When overlap applies, added wall time is much smaller than raw transfer time, though transfers longer than a forward pass cannot be fully hidden; furthermore, if lookup depends on intermediate hidden states rather than just tokens and static tables, it cannot start early at `request.prepare`. Narrow Engram-style tables (like Step 6, width `896`) move two orders of magnitude fewer bytes per row, making host placement proportionally cheaper. Ultimately, table placement trades VRAM for throughput, and `explain` lets you price that tradeoff upfront.

## Step 6: Feeding Lookup Results into the Residual Stream (Engram Style)

So far we added lookup outputs to final-layer logits. What if, like DeepSeek's Engram, you want to inject retrieved vectors into an intermediate layer's residual stream?

In plain terms, Engram hashes context n-grams to look up an embedding table, gates the retrieved vector, and adds it into the residual stream; when the table is huge, it can reside in host memory and be read row-by-row.

Below we build a minimal `EngramLite` using the same corpus and hash function:
- **Table construction (`fit_memory`)**: still `2048` buckets, but instead of `151936`-wide logits bonuses, each bucket stores the word embedding of the next token in the corpus (width equal to the model's hidden size `896`, averaged if a context precedes multiple tokens).
- **State declaration (`memory`)**: both fields use `pllm.Field` (`context` is `(self.buckets, 2)` `int64`, and `memory` uses `lambda width, _: (self.buckets, width)` to read the model's hidden width), so they live in one `InstanceState`.
- **Lookup and injection**: `lookup` reads `pllm.input_tokens(history=1)`, hashes keys, and checks context; `add` (`@pllm.custom_residual(site="attention", layers=(20,))`) adds `self.gate * e` to the residual stream after attention at layer 20.

Two things differ from `NGramPrior`:
1. Because `lookup`'s output feeds `custom_residual`, it runs for **every scheduled token** (including every prompt token), not just sampled rows.
2. `lookup` depends only on input token IDs and the static table (not earlier hidden states). pLLM detects this and executes `lookup` ahead of layer 0 on the `side` CUDA stream.

Let's define `EngramLite` and inspect `lookup`'s scheduling line in `pllm.explain`.

In [9]:
# Read the embedding matrix directly from the model's safetensors on CPU without loading the full model.
EMBEDDING = "model.embed_tokens.weight"
embedding = pllm.io.checkpoint_tensors(pllm.io.checkpoint_directory(MODEL), (EMBEDDING,))[EMBEDDING]


def fit_memory(text: str, buckets: int, path: str) -> None:
    """Engram-style narrow table: each bucket stores the embedding of the next token in the corpus (averaged if multiple)."""
    ids = tokenizer(text).input_ids
    context = torch.full((buckets, 2), -2, dtype=torch.int64)
    rows, counts = torch.zeros(buckets, embedding.shape[1]), torch.zeros(buckets, 1)
    for t in range(1, len(ids) - 1):
        key = int(ngram_keys(torch.tensor(ids[t]), torch.tensor(ids[t - 1]), buckets))
        context[key] = torch.tensor([ids[t], ids[t - 1]])
        rows[key] += embedding[ids[t + 1]].float()
        counts[key] += 1
    save_file({"context": context, "memory": (rows / counts.clamp_min(1)).bfloat16()}, path)


MEMORY = os.path.join(tempfile.gettempdir(), "lantern_memory.safetensors")
fit_memory(corpus, BUCKETS, MEMORY)


@dataclass(frozen=True)
class EngramLite(pllm.Workload):
    path: str = MEMORY
    gate: float = 0.0
    buckets: int = BUCKETS

    @property
    def memory(self) -> pllm.InstanceState:
        return pllm.InstanceState({"context": pllm.Field((self.buckets, 2), torch.int64),
                                   "memory": pllm.Field(lambda width, _: (self.buckets, width))},
                                  init=pllm.Pretrained(self.path))

    def lookup(self, ids: torch.Tensor = pllm.input_tokens(history=1),
               context: torch.Tensor = pllm.parameter("memory", "context"),
               memory: torch.Tensor = pllm.parameter("memory", "memory")) -> torch.Tensor:
        keys = ngram_keys(ids[:, 0], ids[:, 1], self.buckets)
        hit = (context[keys] == ids).all(-1, keepdim=True)
        return torch.where(hit, memory[keys].float(), 0.0)  # [every scheduled token, hidden]

    @pllm.custom_residual(site="attention", layers=(20,))
    def add(self, x: torch.Tensor, y: torch.Tensor, e: torch.Tensor = pllm.result("lookup")) -> torch.Tensor:
        return (x.float() + y.float() + self.gate * e).to(x.dtype)

In [10]:
for line in str(pllm.explain(EngramLite(gate=8.0), CONFIG)).splitlines():
    if line.startswith(("forward", "lookup", "add ")):
        print(line)

forward  CUDA graphs: no effect needs eager
add  custom_residual  model site or inactive method
lookup  compute  request.prepare / side / captured  memory (entry, 0)


In [11]:
for gate in (0.0, 8.0):
    with pllm.make_llm(CONFIG, workload=EngramLite(gate=gate)) as llm:
        for text in answers(llm):
            print(f"gate={gate}:", repr(text))

gate=0.0: 'The Lantern Fair of Quillmere is a traditional festival celebrated in the small town of Quillmere, located in the United States. This event is a significant cultural and economic event that has been held'
gate=0.0: 'The Lantern Fair is a traditional Chinese festival that takes place in the month of Lantern Festival, which is the 15th day of the first lunar month. The Lantern Fair is a time for people'


gate=8.0: 'The Lantern Fair of Quillmere is a traditional festival celebrated in the small town of Quillmere, located in the United States. This event is a significant cultural and economic event that has been held'
gate=8.0: 'The Lantern Fair is a traditional Chinese festival that is celebrated in many parts of China, including the Pearl River Delta region. At the end of the Lantern Fair, there are several significant events that take place'


Look at the three lines from `explain`:
- `forward  CUDA graphs: no effect needs eager`: the model still runs inside CUDA graphs without falling back to eager mode.
- `lookup  compute  request.prepare / side / captured  memory (entry, 0)`: `lookup`'s execution anchor is `request.prepare`, running on the `side` stream and captured in CUDA graphs (`captured`). Before the main stream starts layer 0, the side stream is already performing the table lookup concurrently with the model forward.

Now look at the generated text:
- Raising `gate` from `0.0` to `8.0` leaves the first answer unchanged and alters only the second half of the second answer without reciting the corpus. This matches our upfront caveat: the table only holds raw next-token word embeddings from the corpus, so injecting them into layer 20 of an untrained model has a weak effect; in real Engram, the table and gate are trained jointly with the model.
- You may notice that `gate=0.0` here differs slightly in the second half of the second answer from Step 4's "vanilla vLLM" output. Attaching `custom_residual` captures the model's original operators inside CUDA graphs without Inductor fusion, producing a slightly different floating-point rounding order than default vanilla vLLM (as discussed in Example 09). To verify strict equivalence at `gate=0.0`, compare bitwise against a baseline using the same graph settings—which is exactly what `testing.check_on_gpu` does next.

## Step 7: GPU Contract Checks

Finally, we run `testing.check_on_gpu` on GPU for both `NGramPrior` (`scale=0.0` as `neutral`) and `EngramLite` (`gate=0.0` as `neutral`), verifying four runtime contracts in the real vLLM engine:
1. `explain/probe`: the `explain` plan computed solely from `config.json` matches recomputation after loading the real model;
2. `host syncs`: zero GPU-to-CPU synchronizations over 8 steady decode steps;
3. `preempted`: after request preemption and recomputation, per-token inputs, generated token IDs, and terminal state match identically;
4. `neutral`: at `scale=0.0` or `gate=0.0`, greedy decoding and fixed-seed sampling are bitwise identical to the vanilla model under the same graph settings.

In [12]:
print(testing.check_on_gpu(NGramPrior(scale=10.0), CONFIG, neutral=NGramPrior(scale=0.0)))
print(testing.check_on_gpu(EngramLite(gate=8.0), CONFIG, neutral=EngramLite(gate=0.0)))

status  contract / reason
PASS    explain/probe
PASS    host syncs: eight steady decode steps
PASS    preempted: fixed per-token inputs, token IDs and terminal request state
PASS    neutral: greedy and seeded sampling bitwise


status  contract / reason
PASS    explain/probe
PASS    host syncs: eight steady decode steps
PASS    preempted: fixed per-token inputs, token IDs and terminal request state
PASS    neutral: greedy and seeded sampling bitwise


All four GPU checks `PASS` for both workloads: zero GPU-to-CPU syncs over 8 decode steps (reading token history introduces no sync either), exact reproduction after preemption recompute, and bitwise parity with vanilla vLLM under matching settings when `scale` or `gate` is `0`.

### Summary

Recapping what this guide covered:
- `pllm.input_tokens(history=n)` provides `[current token, previous, ..., n back]` per row directly on GPU without GPU-to-CPU synchronization.
- `InstanceState(..., init=pllm.Pretrained(file_path))` loads tables from safetensors and records the file's `sha256` hash in the workload identity.
- Placement is specified explicitly via `placement="device"` or `placement="host"`; pLLM never silently relocates tables, and `pllm.explain` reports VRAM reservation, host memory footprint, and PCIe bytes/row so you can price `host` placement upfront.
- The same lookup method can feed either `@pllm.add_logits` (per sampled row) or `@pllm.custom_residual` (per scheduled token, overlapping early on the side stream when depending only on token IDs and static tables).
- Whether lookup memory improves quality depends on how the table and model were trained; pLLM handles bringing lookup and injection cleanly into inference.

Importable versions of both workloads live in `examples/engram_lite.py` and `examples/ngram_prior.py` in the repository.

### Next

In the next tutorial, [12 Adaptive Depth](12_adaptive_depth.ipynb), we look at how to let each token decide for itself which model layers to skip.